In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata
import os
import subprocess
import tempfile

token = userdata.get("git_token")

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

askpass = "/tmp/git_askpass.sh"

with open(askpass, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass, 0o700)
env["GIT_ASKPASS"] = askpass

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "dev",
        "https://github.com/brainCDSS/model.git",
        "/content/model",
    ],
    env=env,
    check=True,
)

os.remove(askpass)
del token

print("clone 완료")

clone 완료


In [ ]:
%cd /content/model


/content/model


In [ ]:
!git status -sb

## dev...origin/dev


In [ ]:
!find . \( -iname "*segresnet*" -o -iname "*segres*" \) -type f | sort

./notebooks/04_segresnet_H-S00.ipynb
./notebooks/19_A-S00_SegResNet_Baseline.ipynb


In [ ]:
!find configs/experiments/ischemia -type f | sort

configs/experiments/ischemia/I-N00.yaml
configs/experiments/ischemia/I-S00-smoke.yaml
configs/experiments/ischemia/I-S00.yaml
configs/experiments/ischemia/I-S10.yaml
configs/experiments/ischemia/I-S20.yaml
configs/experiments/ischemia/I-S30.yaml


In [ ]:
!cat configs/experiments/ischemia/I-S00.yaml

extends: I-S00-smoke.yaml
training: {epochs: 300}
model: {features: 16}


In [ ]:
!grep -Rni "SegResNet" src configs --exclude-dir="__pycache__"

src/brainon_ai/models/hemorrhage/segresnet/network.py:4:from monai.networks.nets import SegResNet
src/brainon_ai/models/hemorrhage/segresnet/network.py:13:    """Additive attention for a SegResNet skip connection."""
src/brainon_ai/models/hemorrhage/segresnet/network.py:53:class SegResNetVariant(SegResNet):
src/brainon_ai/models/hemorrhage/segresnet/network.py:90:    name = str(cfg.get("name", "segresnet")).lower()
src/brainon_ai/models/hemorrhage/segresnet/network.py:100:    if name == "segresnet":
src/brainon_ai/models/hemorrhage/segresnet/network.py:101:        return SegResNet(**common)
src/brainon_ai/models/hemorrhage/segresnet/network.py:102:    if name == "segresnet_attention_gate":
src/brainon_ai/models/hemorrhage/segresnet/network.py:103:        return SegResNetVariant(attention=True, multiscale=False, **common)
src/brainon_ai/models/hemorrhage/segresnet/network.py:104:    if name == "segresnet_multiscale_context":
src/brainon_ai/models/hemorrhage/segresnet/network.py:106:    

In [ ]:
!cat configs/experiments/ischemia/I-S00-smoke.yaml

task: ischemia
experiment_id: I-S00
data_release: isles2022_dwi_adc_v1.0.0
split_id: synthetic-v1
data:
  manifest: ${BRAINON_CACHE_ROOT:-.cache/brainon}/synthetic/ischemia/manifest.csv
model: {name: segresnet_early_fusion, in_channels: 2, features: 4}
runtime: {seed: 42, device: cpu, amp: false, deterministic: true}
training: {epochs: 1, learning_rate: 0.001, gradient_accumulation: 1, early_stopping_patience: 3, resume: null}
inference: {sliding_window: true, roi_size: [16, 16, 16], overlap: 0.25}
evaluation: {threshold: 0.5, postprocessing: {minimum_component_size: 0}}
selection:
  primary: val/lesion_f1_mean
  primary_mode: max
  primary_tolerance: 0.01
  secondary: val/dice_fg_case_mean
  secondary_mode: max
  tie_breaker: val/avd_ml_median



In [ ]:
!find src/brainon_ai -path "*ischemia*" -type f | sort

src/brainon_ai/data/ischemia/__init__.py
src/brainon_ai/models/ischemia/__init__.py


In [ ]:
!pwd
!ls -la /content/model/configs/experiments/ischemia/

/content/model
total 32
drwxr-xr-x 2 root root 4096 Sep  3 02:13 .
drwxr-xr-x 5 root root 4096 Sep  3 02:13 ..
-rw-r--r-- 1 root root  747 Sep  3 02:13 I-N00.yaml
-rw-r--r-- 1 root root  756 Sep  3 02:13 I-S00-smoke.yaml
-rw-r--r-- 1 root root   72 Sep  3 02:13 I-S00.yaml
-rw-r--r-- 1 root root   69 Sep  3 02:13 I-S10.yaml
-rw-r--r-- 1 root root   74 Sep  3 02:13 I-S20.yaml
-rw-r--r-- 1 root root   81 Sep  3 02:13 I-S30.yaml


In [ ]:
!cat /content/model/configs/experiments/ischemia/I-S00-smoke.yaml

task: ischemia
experiment_id: I-S00
data_release: isles2022_dwi_adc_v1.0.0
split_id: synthetic-v1
data:
  manifest: ${BRAINON_CACHE_ROOT:-.cache/brainon}/synthetic/ischemia/manifest.csv
model: {name: segresnet_early_fusion, in_channels: 2, features: 4}
runtime: {seed: 42, device: cpu, amp: false, deterministic: true}
training: {epochs: 1, learning_rate: 0.001, gradient_accumulation: 1, early_stopping_patience: 3, resume: null}
inference: {sliding_window: true, roi_size: [16, 16, 16], overlap: 0.25}
evaluation: {threshold: 0.5, postprocessing: {minimum_component_size: 0}}
selection:
  primary: val/lesion_f1_mean
  primary_mode: max
  primary_tolerance: 0.01
  secondary: val/dice_fg_case_mean
  secondary_mode: max
  tie_breaker: val/avd_ml_median



In [ ]:
!cat /content/model/configs/experiments/ischemia/I-N00.yaml

task: ischemia
experiment_id: I-N00

data:
  dataset_id: 502
  dataset_name: Dataset502_ISLES2022
  modalities:
    - DWI
    - ADC
  in_channels: 2

split:
  fold: 0
  train_cases: 175
  validation_cases: 38
  test_cases: 37

model:
  name: nnunet
  trainer: nnUNetTrainerIschemiaBaseline
  configuration: 3d_fullres
  plans: nnUNetPlans

preprocessing:
  spacing:
    - 2.0
    - 2.0
    - 2.0
  patch_size:
    - 80
    - 96
    - 80
  batch_size: 6
  normalization:
    - ZScore
    - ZScore

training:
  epochs: 300
  device: cuda
  early_stopping:
    enabled: true
    monitor: val_dice
    mode: max
    patience: 30
    min_delta: 0.001

checkpoint:
  save_best: true
  save_latest_every: 1
  monitor: val_dice
  mode: max
  resume: true


In [ ]:
!cat /content/model/registry/splits/I-N00_v1.0.0.csv | head

case_id,split
sub-strokecase0045,train
sub-strokecase0099,train
sub-strokecase0185,train
sub-strokecase0124,train
sub-strokecase0122,train
sub-strokecase0168,train
sub-strokecase0010,train
sub-strokecase0177,train
sub-strokecase0198,train


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!gcloud storage ls gs://brainon-cdss-datasets/ischemia/v1/SegResNet/

gs://brainon-cdss-datasets/ischemia/v1/SegResNet/metadata/
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/test/
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/val/


In [ ]:
!gcloud storage ls gs://brainon-cdss-datasets/ischemia/v1/SegResNet/metadata/

gs://brainon-cdss-datasets/ischemia/v1/SegResNet/metadata/preprocessing_config.json


In [ ]:
!echo "TRAIN"
!gcloud storage ls --recursive gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/ | grep -E '\.nii(\.gz)?$' | wc -l

!echo "VAL"
!gcloud storage ls --recursive gs://brainon-cdss-datasets/ischemia/v1/SegResNet/val/ | grep -E '\.nii(\.gz)?$' | wc -l

!echo "TEST"
!gcloud storage ls --recursive gs://brainon-cdss-datasets/ischemia/v1/SegResNet/test/ | grep -E '\.nii(\.gz)?$' | wc -l

TRAIN
525
VAL
114
TEST
111


In [ ]:
!gcloud storage ls --recursive gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/ | head -20

gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/:

gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/:
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0001_adc.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0001_dwi.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0003_adc.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0003_dwi.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0004_adc.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0004_dwi.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0005_adc.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0005_dwi.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0006_adc.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/t

In [ ]:
# gcs SegResNet 전처리 데이터 Train/Val/Test별 DWI, ADC, Mask 파일 수 검증
%%bash

for SPLIT in train val test; do
  echo "===== $SPLIT ====="

  echo -n "ADC  : "
  gcloud storage ls \
    "gs://brainon-cdss-datasets/ischemia/v1/SegResNet/$SPLIT/images/*_adc.nii.gz" | wc -l

  echo -n "DWI  : "
  gcloud storage ls \
    "gs://brainon-cdss-datasets/ischemia/v1/SegResNet/$SPLIT/images/*_dwi.nii.gz" | wc -l

  echo -n "MASK : "
  gcloud storage ls \
    "gs://brainon-cdss-datasets/ischemia/v1/SegResNet/$SPLIT/labels/*_mask.nii.gz" | wc -l
done

===== train =====
ADC  : 175
DWI  : 175
MASK : 175
===== val =====
ADC  : 38
DWI  : 38
MASK : 38
===== test =====
ADC  : 37
DWI  : 37
MASK : 37


In [ ]:
# GCS SegResNet Train 데이터의 Mask 저장 경로 확인

!gcloud storage ls --recursive \
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/ \
| grep -E '\.nii\.gz$' \
| grep -v '/images/' \
| head -20

gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0001_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0003_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0004_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0005_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0006_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0010_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0011_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0013_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0014_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-strokecase0015_mask.nii.gz
gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/labels/sub-stro

In [ ]:
# GCS SegResNet 전처리 데이터를 Colab 학습 공간으로 복사

%%bash

mkdir -p /content/segresnet_data

gcloud storage cp --recursive \
  gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train \
  gs://brainon-cdss-datasets/ischemia/v1/SegResNet/val \
  gs://brainon-cdss-datasets/ischemia/v1/SegResNet/test \
  gs://brainon-cdss-datasets/ischemia/v1/SegResNet/metadata \
  /content/segresnet_data/

Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0001_adc.nii.gz to file:///content/segresnet_data/train/images/sub-strokecase0001_adc.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0001_dwi.nii.gz to file:///content/segresnet_data/train/images/sub-strokecase0001_dwi.nii.gz
  
Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0003_adc.nii.gz to file:///content/segresnet_data/train/images/sub-strokecase0003_adc.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0003_dwi.nii.gz to file:///content/segresnet_data/train/images/sub-strokecase0003_dwi.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0004_adc.nii.gz to file:///content/segresnet_data/train/images/sub-strokecase0004_adc.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train/images/sub-strokecase0004_dwi.nii.gz to file:///c

In [ ]:
# Colab에 복사된 SegResNet 전처리 데이터 최종 개수 검증

from pathlib import Path

root = Path("/content/segresnet_data")

for split in ["train", "val", "test"]:
    adc = list((root / split / "images").glob("*_adc.nii.gz"))
    dwi = list((root / split / "images").glob("*_dwi.nii.gz"))
    mask = list((root / split / "labels").glob("*_mask.nii.gz"))

    print(
        f"{split.upper():5s} | "
        f"ADC: {len(adc):3d} | "
        f"DWI: {len(dwi):3d} | "
        f"MASK: {len(mask):3d}"
    )

TRAIN | ADC: 175 | DWI: 175 | MASK: 175
VAL   | ADC:  38 | DWI:  38 | MASK:  38
TEST  | ADC:  37 | DWI:  37 | MASK:  37


In [ ]:
# SegResNet 학습 파일을 저장할 기존 저장소 구조 확인
!echo "===== registry ====="
!find /content/model/registry -maxdepth 2 -type f | sort | head -50

!echo ""
!echo "===== ischemia model ====="
!find /content/model/src/brainon_ai/models/ischemia -maxdepth 3 -type f | sort

!echo ""
!echo "===== preprocessing metadata ====="
!cat /content/segresnet_data/metadata/preprocessing_config.json

===== registry =====
/content/model/registry/datasets/hemorrhage_binary_v1.0.0.yaml
/content/model/registry/datasets/isles2022_dwi_adc_v1.0.0.yaml
/content/model/registry/datasets/topaneu2026_mra_v1.0.0.yaml
/content/model/registry/manifests/README.md
/content/model/registry/models/README.md
/content/model/registry/splits/I-N00_v1.0.0.csv
/content/model/registry/splits/README.md

===== ischemia model =====
/content/model/src/brainon_ai/models/ischemia/__init__.py
/content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py
/content/model/src/brainon_ai/models/ischemia/segresnet/__init__.py

===== preprocessing metadata =====
{
  "dataset": "ISLES2022",
  "case_count": 250,
  "excluded_cases": [],
  "modalities": [
    "DWI",
    "ADC"
  ],
  "label": {
    "name": "Lesion Mask",
    "background": 0,
    "lesion": 1
  },
  "target_spacing_mm": [
    2.0,
    2.0,
    2.0
  ],
  "reference_grid": "DWI",
  "resampling": {
    "DWI": {
      "interpolation": "linear"
    },
    "ADC

In [ ]:
# ISLES 2022 SegResNet 학습용 manifest 생성
# DWI + ADC + Mask 경로와 Train/Val/Test split 정보를 하나의 CSV로 관리

from pathlib import Path
import pandas as pd

data_root = Path("/content/segresnet_data")
manifest_path = Path(
    "/content/model/registry/manifests/I-S00_v1.0.0.csv"
)

rows = []

for split_folder, split_name in [
    ("train", "train"),
    ("val", "validation"),
    ("test", "test"),
]:
    image_dir = data_root / split_folder / "images"
    label_dir = data_root / split_folder / "labels"

    for dwi_path in sorted(image_dir.glob("*_dwi.nii.gz")):
        case_id = dwi_path.name.replace("_dwi.nii.gz", "")

        adc_path = image_dir / f"{case_id}_adc.nii.gz"
        mask_path = label_dir / f"{case_id}_mask.nii.gz"

        if not adc_path.is_file():
            raise FileNotFoundError(f"ADC 없음: {case_id}")

        if not mask_path.is_file():
            raise FileNotFoundError(f"Mask 없음: {case_id}")

        rows.append({
            "case_id": case_id,
            "split": split_name,
            "dwi_path": str(dwi_path),
            "adc_path": str(adc_path),
            "mask_path": str(mask_path),
        })

manifest = pd.DataFrame(rows)
manifest.to_csv(manifest_path, index=False)

print(f"Manifest 저장 완료: {manifest_path}")
print()
print(manifest.groupby("split").size())
print()
print(f"전체 Case: {len(manifest)}")
print()
print(manifest.head())

Manifest 저장 완료: /content/model/registry/manifests/I-S00_v1.0.0.csv

split
test           37
train         175
validation     38
dtype: int64

전체 Case: 250

              case_id  split  \
0  sub-strokecase0001  train   
1  sub-strokecase0003  train   
2  sub-strokecase0004  train   
3  sub-strokecase0005  train   
4  sub-strokecase0006  train   

                                            dwi_path  \
0  /content/segresnet_data/train/images/sub-strok...   
1  /content/segresnet_data/train/images/sub-strok...   
2  /content/segresnet_data/train/images/sub-strok...   
3  /content/segresnet_data/train/images/sub-strok...   
4  /content/segresnet_data/train/images/sub-strok...   

                                            adc_path  \
0  /content/segresnet_data/train/images/sub-strok...   
1  /content/segresnet_data/train/images/sub-strok...   
2  /content/segresnet_data/train/images/sub-strok...   
3  /content/segresnet_data/train/images/sub-strok...   
4  /content/segresnet_data/train/i

In [ ]:
# SegResNet runtime manifest 이동 및 nnU-Net과 동일 split 여부 검증

from pathlib import Path
import pandas as pd
import shutil

repo_manifest = Path("/content/model/registry/manifests/I-S00_v1.0.0.csv")
runtime_manifest = Path("/content/segresnet_data/metadata/I-S00_runtime_manifest.csv")

# runtime 전용 위치로 이동
shutil.move(str(repo_manifest), str(runtime_manifest))

# SegResNet manifest
seg = pd.read_csv(runtime_manifest)[["case_id", "split"]].copy()

# nnU-Net에서 사용한 split
nn = pd.read_csv(
    "/content/model/registry/splits/I-N00_v1.0.0.csv"
)[["case_id", "split"]].copy()

seg = seg.sort_values("case_id").reset_index(drop=True)
nn = nn.sort_values("case_id").reset_index(drop=True)

print("SegResNet split:")
print(seg["split"].value_counts())
print()

print("nnU-Net split:")
print(nn["split"].value_counts())
print()

print("전체 Case 동일:", set(seg.case_id) == set(nn.case_id))
print("Case별 Split 동일:", seg.equals(nn))

if not seg.equals(nn):
    diff = seg.merge(nn, on="case_id", suffixes=("_seg", "_nn"))
    diff = diff[diff["split_seg"] != diff["split_nn"]]
    print("\n불일치 Case:")
    print(diff)
else:
    print("\n✓ nnU-Net과 SegResNet이 동일한 Train/Validation/Test split을 사용합니다.")

print("\nRuntime manifest:", runtime_manifest)

SegResNet split:
split
train         175
validation     38
test           37
Name: count, dtype: int64

nnU-Net split:
split
train         175
validation     38
test           37
Name: count, dtype: int64

전체 Case 동일: True
Case별 Split 동일: True

✓ nnU-Net과 SegResNet이 동일한 Train/Validation/Test split을 사용합니다.

Runtime manifest: /content/segresnet_data/metadata/I-S00_runtime_manifest.csv


In [ ]:
# SegResNet 학습용 GPU 확인

import torch

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

CUDA 사용 가능: True
GPU: Tesla T4
VRAM: 14.6 GB


In [ ]:
# SegResNet 학습 환경 설치
!pip install -q monai nibabel pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.5 MB/s eta 0:00:00


In [ ]:
import torch
import monai

print("PyTorch:", torch.__version__)
print("MONAI:", monai.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
MONAI: 1.6.0
GPU: Tesla T4


In [ ]:
# 허혈성 SegResNet 학습 코드 디렉터리 생성

from pathlib import Path

segresnet_dir = Path(
    "/content/model/src/brainon_ai/models/ischemia/segresnet"
)

segresnet_dir.mkdir(parents=True, exist_ok=True)

(segresnet_dir / "__init__.py").touch()

print("생성 완료:", segresnet_dir)

생성 완료: /content/model/src/brainon_ai/models/ischemia/segresnet


In [ ]:
# SegResNet checkpoint 영구 백업을 위한 Google Drive 연결

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# SegResNet I-S00 학습 결과 및 checkpoint 영구 백업 경로 생성

from pathlib import Path

backup_root = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00"
)

checkpoint_dir = backup_root / "checkpoints"
log_dir = backup_root / "logs"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

print("백업 경로:", backup_root)
print("Checkpoint:", checkpoint_dir)
print("Log:", log_dir)

백업 경로: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00
Checkpoint: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/checkpoints
Log: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/logs


In [ ]:


!python /content/model/src/brainon_ai/models/ischemia/segresnet/train.py \
  --config /content/model/configs/experiments/ischemia/I-S00.yaml \
  --data-root /content/segresnet_data \
  --backup-root /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00

I-S00 SegResNet Baseline
GPU : Tesla T4
Train: 175
Val  : 38
Test : 37
입력 : DWI + ADC (2 channels)
Patch: (80, 96, 80)
/content/model/src/brainon_ai/models/ischemia/segresnet/train.py:293: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)
/usr/local/lib/python3.13/dist-packages/monai/transforms/utils.py:682: UserWarning: Num foregrounds 0, Num backgrounds 1003520, unable to generate class balanced samples, setting `pos_ratio` to 0.
  warnings.warn(
/content/model/src/brainon_ai/models/ischemia/segresnet/train.py:392: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/content/model/src/brainon_ai/models/ischemia/segresnet/train.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)

In [ ]:
from pathlib import Path

p = Path("/content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py")

if p.exists():
    print(p.read_text())
else:
    print("evaluate.py 파일이 없습니다.")


from __future__ import annotations

import json
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import torch

from monai.data import DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet
from monai.transforms import (
    Compose,
    ConcatItemsd,
    DeleteItemsd,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
)

from scipy.ndimage import (
    binary_erosion,
    distance_transform_edt,
    label as cc_label,
)


# ============================================================
# 데이터
# ============================================================

def build_records(root: Path, split: str):
    images = root / split / "images"
    labels = root / split / "labels"

    records = []

    for dwi in sorted(images.glob("*_dwi.nii.gz")):
        case_id = dwi.name.replace("_dwi.nii.gz", "")

        adc = images / f"{case_id}_adc.nii.gz"
        mask = labels / f"{case_id}_mask.nii.g

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00"
)

CHECKPOINT = ROOT / "checkpoints/checkpoint_best.pth"

print("===== SegResNet Validation 평가 준비 =====")
print("checkpoint_best.pth:", CHECKPOINT.is_file())
print("경로:", CHECKPOINT)

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"checkpoint_best.pth가 없습니다:\n{CHECKPOINT}"
    )

print("\n✓ 기존 Best Checkpoint 확인 완료")
print("✓ 학습 재실행 없이 Validation 38명 평가에 사용합니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
===== SegResNet Validation 평가 준비 =====
checkpoint_best.pth: True
경로: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/checkpoints/checkpoint_best.pth

✓ 기존 Best Checkpoint 확인 완료
✓ 학습 재실행 없이 Validation 38명 평가에 사용합니다.


In [ ]:
# ============================================================
# I-S00 SegResNet Validation 상세 평가
#
# 목적
# - 학습 다시 하지 않음
# - 기존 checkpoint_best.pth 사용
# - Validation 38 case 평가
# - 기존 TEST 결과 절대 수정하지 않음
#
# 저장 결과
# validation/
# ├─ validation_patient_level_metrics.csv
# ├─ validation_summary_metrics.csv
# ├─ validation_evaluation_metadata.json
# └─ validation_predictions/
# ============================================================

from pathlib import Path
import importlib.util
import inspect
import math
import json

import numpy as np
import pandas as pd
import torch
import nibabel as nib

from monai.data import Dataset, DataLoader
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet


# ============================================================
# 1. 경로 설정
# ============================================================

SCRIPT = Path(
    "/content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py"
)

DATA_ROOT = Path(
    "/content/segresnet_data"
)

ROOT = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00"
)

CHECKPOINT = (
    ROOT
    / "checkpoints"
    / "checkpoint_best.pth"
)

# 기존 test evaluation과 완전히 분리
VALIDATION_DIR = (
    ROOT
    / "evaluation"
    / "validation"
)

PREDICTION_DIR = (
    VALIDATION_DIR
    / "validation_predictions"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. 필수 파일 확인
# ============================================================

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"evaluate.py가 없습니다:\n{SCRIPT}"
    )

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"checkpoint_best.pth가 없습니다:\n{CHECKPOINT}"
    )

VAL_ROOT = DATA_ROOT / "val"

counts = {
    "DWI": len(
        list(
            (VAL_ROOT / "images")
            .glob("*_dwi.nii.gz")
        )
    ),
    "ADC": len(
        list(
            (VAL_ROOT / "images")
            .glob("*_adc.nii.gz")
        )
    ),
    "MASK": len(
        list(
            (VAL_ROOT / "labels")
            .glob("*_mask.nii.gz")
        )
    ),
}

print("===== Validation 데이터 확인 =====")
print(counts)

if counts != {
    "DWI": 38,
    "ADC": 38,
    "MASK": 38,
}:
    raise RuntimeError(
        "Validation 데이터가 각각 38개가 아닙니다.\n"
        "런타임이 끊어진 경우 GCS 데이터를 "
        "/content/segresnet_data 로 다시 복사해야 합니다."
    )


# ============================================================
# 3. GPU 확인
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 연결되지 않았습니다.\n"
        "Colab → 런타임 → 런타임 유형 변경 → T4 GPU 선택"
    )

device = torch.device("cuda")

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# 4. 기존 evaluate.py 불러오기
# ============================================================

spec = importlib.util.spec_from_file_location(
    "segresnet_evaluation",
    SCRIPT,
)

evaluation = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    evaluation
)


# ------------------------------------------------------------
# evaluate.py MCC 계산 보정
# 원본 evaluate.py 파일 자체는 수정하지 않음
# ------------------------------------------------------------

voxel_source = inspect.getsource(
    evaluation.calculate_voxel_metrics
)

if "np.sqrt(" in voxel_source:

    evaluation.__dict__["math"] = math

    exec(
        voxel_source.replace(
            "np.sqrt(",
            "math.sqrt("
        ),
        evaluation.__dict__,
    )


# ============================================================
# 5. checkpoint_best.pth 로드
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=False,
)

cfg = checkpoint["config"]

best_epoch = (
    int(checkpoint["epoch"])
    + 1
)

best_validation_dice = float(
    checkpoint["best_dice"]
)

patch_size = tuple(
    cfg["data"]["patch_size"]
)

threshold = float(
    cfg["evaluation"]["threshold"]
)

sw_batch_size = int(
    cfg["evaluation"]["sw_batch_size"]
)

overlap = float(
    cfg["evaluation"]["overlap"]
)

amp = bool(
    cfg["runtime"]["amp"]
)


# ============================================================
# 6. 동일한 SegResNet 모델 생성
# ============================================================

model = SegResNet(
    spatial_dims=3,

    init_filters=int(
        cfg["model"]["init_filters"]
    ),

    in_channels=int(
        cfg["data"]["in_channels"]
    ),

    out_channels=int(
        cfg["model"]["out_channels"]
    ),

    dropout_prob=0.0,
).to(device)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()


# ============================================================
# 7. Validation 38 case Dataset
# ============================================================

val_records = evaluation.build_records(
    DATA_ROOT,
    "val",
)

if len(val_records) != 38:
    raise RuntimeError(
        f"Validation Case 수 오류: "
        f"{len(val_records)} "
        f"(예상 38)"
    )

val_dataset = Dataset(
    val_records,
    transform=evaluation.test_transform(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


print()
print("=" * 75)
print("I-S00 SegResNet VALIDATION Evaluation")
print("=" * 75)

print("Checkpoint       :", CHECKPOINT)
print("Best Epoch       :", best_epoch)

print(
    "Training Best Val Dice:",
    f"{best_validation_dice:.6f}"
)

print(
    "Validation Cases :",
    len(val_records)
)

print("Threshold        :", threshold)
print("ROI Size         :", patch_size)
print("SW Batch Size    :", sw_batch_size)
print("Overlap          :", overlap)

print("=" * 75)


# ============================================================
# 8. Validation 환자별 추론 + 성능지표
# ============================================================

patient_results = []


with torch.no_grad():

    for index, batch in enumerate(
        val_loader,
        start=1,
    ):

        # ----------------------------------------------------
        # case_id
        # ----------------------------------------------------

        case_value = batch["case_id"]

        if isinstance(
            case_value,
            (list, tuple),
        ):
            case_id = case_value[0]
        else:
            case_id = str(
                case_value
            )


        # ----------------------------------------------------
        # label path
        # ----------------------------------------------------

        label_path_value = (
            batch["label_path"]
        )

        if isinstance(
            label_path_value,
            (list, tuple),
        ):
            label_path = (
                label_path_value[0]
            )
        else:
            label_path = str(
                label_path_value
            )


        # ----------------------------------------------------
        # GPU
        # ----------------------------------------------------

        image = batch["image"].to(
            device,
            non_blocking=True,
        )

        label = batch["label"].to(
            device,
            non_blocking=True,
        )


        # ----------------------------------------------------
        # Sliding Window Inference
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp,
        ):

            logits = (
                sliding_window_inference(
                    image,
                    roi_size=patch_size,
                    sw_batch_size=sw_batch_size,
                    predictor=model,
                    overlap=overlap,
                )
            )


        probability = torch.sigmoid(
            logits
        )

        prediction = (
            probability
            >= threshold
        )


        # ----------------------------------------------------
        # numpy
        # ----------------------------------------------------

        pred_np = (
            prediction[0, 0]
            .cpu()
            .numpy()
            .astype(np.uint8)
        )

        gt_np = (
            (label[0, 0] > 0.5)
            .cpu()
            .numpy()
            .astype(np.uint8)
        )


        # ----------------------------------------------------
        # spacing / geometry
        # ----------------------------------------------------

        gt_nifti = nib.load(
            label_path
        )

        spacing = tuple(
            float(x)
            for x in
            gt_nifti.header.get_zooms()[:3]
        )


        # ----------------------------------------------------
        # 병변 상태
        # ----------------------------------------------------

        gt_exists = bool(
            gt_np.any()
        )

        pred_exists = bool(
            pred_np.any()
        )

        if gt_exists and pred_exists:

            case_status = (
                "GT_POS_PRED_POS"
            )

        elif gt_exists and not pred_exists:

            case_status = (
                "GT_POS_PRED_EMPTY"
            )

        elif (
            not gt_exists
            and pred_exists
        ):

            case_status = (
                "GT_EMPTY_PRED_POS"
            )

        else:

            case_status = (
                "GT_EMPTY_PRED_EMPTY"
            )


        # ----------------------------------------------------
        # 기존 evaluate.py metric 함수 그대로 사용
        # ----------------------------------------------------

        voxel_metrics = (
            evaluation
            .calculate_voxel_metrics(
                pred_np,
                gt_np,
            )
        )

        surface_metrics = (
            evaluation
            .calculate_surface_metrics(
                pred_np,
                gt_np,
                spacing,
            )
        )

        volume_metrics = (
            evaluation
            .calculate_volume_metrics(
                pred_np,
                gt_np,
                spacing,
            )
        )

        lesion_metrics = (
            evaluation
            .calculate_lesion_metrics(
                pred_np,
                gt_np,
            )
        )


        # ----------------------------------------------------
        # Prediction mask 저장
        # ----------------------------------------------------

        pred_path = (
            PREDICTION_DIR
            / f"{case_id}_pred_mask.nii.gz"
        )

        header = (
            gt_nifti.header.copy()
        )

        header.set_data_dtype(
            np.uint8
        )

        pred_nifti = (
            nib.Nifti1Image(
                pred_np,
                affine=gt_nifti.affine,
                header=header,
            )
        )

        nib.save(
            pred_nifti,
            pred_path,
        )


        # ----------------------------------------------------
        # 환자별 결과
        # ----------------------------------------------------

        result = {
            "case_id":
                case_id,

            "case_status":
                case_status,

            "spacing_x_mm":
                spacing[0],

            "spacing_y_mm":
                spacing[1],

            "spacing_z_mm":
                spacing[2],

            **voxel_metrics,
            **surface_metrics,
            **volume_metrics,
            **lesion_metrics,

            "prediction_path":
                str(pred_path),
        }

        patient_results.append(
            result
        )


        hd95 = result[
            "hd95_mm"
        ]

        hd95_text = (
            f"{hd95:.2f}"
            if np.isfinite(hd95)
            else "NA"
        )

        print(
            f"[{index:02d}/38] "
            f"{case_id} | "
            f"Dice={result['dice']:.4f} | "
            f"IoU={result['iou']:.4f} | "
            f"Precision={result['precision']:.4f} | "
            f"Recall={result['recall_sensitivity']:.4f} | "
            f"HD95={hd95_text}"
        )


# ============================================================
# 9. 환자별 CSV 저장
# ============================================================

patient_df = pd.DataFrame(
    patient_results
)

patient_csv = (
    VALIDATION_DIR
    / "validation_patient_level_metrics.csv"
)

patient_df.to_csv(
    patient_csv,
    index=False,
)


# ============================================================
# 10. Validation summary
# ============================================================

metric_columns = [
    "dice",
    "iou",
    "precision",
    "recall_sensitivity",
    "specificity",
    "npv",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "mcc",

    "hd95_mm",
    "hausdorff_max_mm",
    "assd_mm",

    "gt_volume_ml",
    "pred_volume_ml",
    "abs_volume_error_ml",
    "relative_volume_error_pct",
    "volume_similarity",

    "lesion_precision",
    "lesion_recall",
    "lesion_f1",
]


summary_rows = []


for metric in metric_columns:

    values = pd.to_numeric(
        patient_df[metric],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    valid = values[
        np.isfinite(values)
    ]

    if len(valid) == 0:
        continue

    ci_low, ci_high = (
        evaluation.bootstrap_mean_ci(
            valid,
            n_bootstrap=10000,
            seed=42,
        )
    )

    summary_rows.append(
        {
            "metric": metric,

            "n_valid":
                len(valid),

            "mean":
                float(
                    np.mean(valid)
                ),

            "std":
                float(
                    np.std(
                        valid,
                        ddof=1,
                    )
                )
                if len(valid) > 1
                else 0.0,

            "median":
                float(
                    np.median(valid)
                ),

            "q1":
                float(
                    np.percentile(
                        valid,
                        25,
                    )
                ),

            "q3":
                float(
                    np.percentile(
                        valid,
                        75,
                    )
                ),

            "min":
                float(
                    np.min(valid)
                ),

            "max":
                float(
                    np.max(valid)
                ),

            "mean_95ci_low":
                ci_low,

            "mean_95ci_high":
                ci_high,
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    VALIDATION_DIR
    / "validation_summary_metrics.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False,
)


# ============================================================
# 11. Macro Dice 계산
# ============================================================

# Validation 38 case 전체
macro_dice_all = float(
    patient_df["dice"].mean()
)

median_dice_all = float(
    patient_df["dice"].median()
)

std_dice_all = float(
    patient_df["dice"].std(
        ddof=1
    )
)


# GT 병변 존재 case만
positive_df = patient_df[
    patient_df["gt_volume_ml"] > 0
]

macro_dice_positive = float(
    positive_df["dice"].mean()
)


# Empty GT
empty_gt_df = patient_df[
    patient_df["gt_volume_ml"] == 0
]


# ============================================================
# 12. Metadata 저장
# ============================================================

metadata = {

    "experiment_id":
        "I-S00",

    "evaluation_split":
        "validation",

    "checkpoint":
        str(CHECKPOINT),

    "best_epoch":
        best_epoch,

    "training_best_validation_dice":
        best_validation_dice,

    "validation_cases":
        len(patient_df),

    "gt_positive_cases":
        len(positive_df),

    "gt_empty_cases":
        len(empty_gt_df),

    "validation_macro_dice_all":
        macro_dice_all,

    "validation_median_dice_all":
        median_dice_all,

    "validation_std_dice_all":
        std_dice_all,

    "validation_macro_dice_gt_positive_only":
        macro_dice_positive,

    "threshold":
        threshold,

    "roi_size":
        list(patch_size),

    "sw_batch_size":
        sw_batch_size,

    "overlap":
        overlap,
}


metadata_path = (
    VALIDATION_DIR
    / "validation_evaluation_metadata.json"
)

with metadata_path.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. 최종 결과 출력
# ============================================================

print()
print("=" * 75)
print("SEGRESNET VALIDATION 평가 완료")
print("=" * 75)

print(
    f"Validation Case 수             : "
    f"{len(patient_df)}"
)

print(
    f"GT-positive Case 수            : "
    f"{len(positive_df)}"
)

print(
    f"GT-empty Case 수               : "
    f"{len(empty_gt_df)}"
)

print()

print(
    f"Validation Macro Dice (38)     : "
    f"{macro_dice_all:.6f}"
)

print(
    f"Validation Dice Median         : "
    f"{median_dice_all:.6f}"
)

print(
    f"Validation Dice Std            : "
    f"{std_dice_all:.6f}"
)

print(
    f"GT-positive Macro Dice         : "
    f"{macro_dice_positive:.6f}"
)

print()

print(
    f"Checkpoint 기록 Best Val Dice : "
    f"{best_validation_dice:.6f}"
)

print(
    f"재평가와 checkpoint 차이       : "
    f"{macro_dice_all - best_validation_dice:+.8f}"
)

print()
print("저장 파일:")
print(patient_csv)
print(summary_csv)
print(metadata_path)
print(PREDICTION_DIR)

print("=" * 75)


# ============================================================
# 14. 핵심 성능지표만 표로 확인
# ============================================================

important_metrics = [
    "dice",
    "iou",
    "precision",
    "recall_sensitivity",
    "hd95_mm",
    "assd_mm",
]

display(
    summary_df[
        summary_df["metric"].isin(
            important_metrics
        )
    ][
        [
            "metric",
            "n_valid",
            "mean",
            "std",
            "median",
            "mean_95ci_low",
            "mean_95ci_high",
        ]
    ].round(4)
)

===== Validation 데이터 확인 =====
{'DWI': 38, 'ADC': 38, 'MASK': 38}
GPU: Tesla T4

I-S00 SegResNet VALIDATION Evaluation
Checkpoint       : /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/checkpoints/checkpoint_best.pth
Best Epoch       : 104
Training Best Val Dice: 0.770057
Validation Cases : 38
Threshold        : 0.5
ROI Size         : (80, 96, 80)
SW Batch Size    : 2
Overlap          : 0.25
[01/38] sub-strokecase0002 | Dice=0.7892 | IoU=0.6519 | Precision=0.8381 | Recall=0.7458 | HD95=2.00
[02/38] sub-strokecase0012 | Dice=0.7259 | IoU=0.5698 | Precision=0.6203 | Recall=0.8750 | HD95=2.83
[03/38] sub-strokecase0018 | Dice=0.7671 | IoU=0.6222 | Precision=0.9767 | Recall=0.6316 | HD95=2.83
[04/38] sub-strokecase0021 | Dice=0.7952 | IoU=0.6601 | Precision=0.8391 | Recall=0.7558 | HD95=26.98
[05/38] sub-strokecase0025 | Dice=0.8480 | IoU=0.7362 | Precision=0.7583 | Recall=0.9618 | HD95=2.00
[06/38] sub-strokecase0032 | Dice=0.9048 | IoU=0.8261 | Precision=0.9661 | Recall=0.8507 

,metric,n_valid,mean,std,median,mean_95ci_low,mean_95ci_high
0,dice,38,0.7701,0.1570,0.8107,0.7183,0.8159
1,iou,38,0.6482,0.1805,0.6820,0.5904,0.7022
2,precision,38,0.8551,0.1248,0.9044,0.8148,0.8930
3,recall_sensitivity,38,0.7504,0.2108,0.8160,0.6810,0.8123
10,hd95_mm,38,10.3931,14.0506,2.8284,6.2750,14.9528
12,assd_mm,38,2.0956,4.3722,0.7688,1.0929,3.6814


Validation 38명 중 병변이 있는 37명에서 병변을 아예 검출하지 못한 환자 단위 미탐은 없었습니다. 다만 Recall 평균이 0.7504라서, 병변을 검출은 했지만 일부 환자에서 전체 병변 영역을 충분히 segmentation하지 못한 경우는 있습니다

In [ ]:
from pathlib import Path
import shutil

# 현재 Colab 런타임 파일
SRC = Path(
    "/content/segresnet_data/metadata/I-S00_runtime_manifest.csv"
)

# Google Drive 영구 보존 위치
BACKUP_DIR = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/reproducibility"
)

DST = BACKUP_DIR / "I-S00_runtime_manifest.csv"

if not SRC.is_file():
    raise FileNotFoundError(
        f"runtime manifest가 없습니다:\n{SRC}"
    )

BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    SRC,
    DST
)

print("===== Runtime Manifest 백업 완료 =====")
print("원본 :", SRC)
print("Drive:", DST)
print("크기 :", f"{DST.stat().st_size:,} bytes")

if DST.stat().st_size != SRC.stat().st_size:
    raise RuntimeError("백업 파일 크기가 원본과 다릅니다.")

print("✓ Drive 영구 보존 확인 완료")

===== Runtime Manifest 백업 완료 =====
원본 : /content/segresnet_data/metadata/I-S00_runtime_manifest.csv
Drive: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/reproducibility/I-S00_runtime_manifest.csv
크기 : 56,606 bytes
✓ Drive 영구 보존 확인 완료


In [ ]:
from pathlib import Path
import shutil

# 현재 Validation 평가에 실제 사용한 evaluate.py
SRC = Path(
    "/content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py"
)

# Google Drive 영구 보존 위치
BACKUP_DIR = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/reproducibility"
)

DST = BACKUP_DIR / "evaluate.py"

if not SRC.is_file():
    raise FileNotFoundError(
        f"evaluate.py가 없습니다:\n{SRC}"
    )

BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    SRC,
    DST
)

print("===== evaluate.py 백업 완료 =====")
print("원본 :", SRC)
print("Drive:", DST)
print("크기 :", f"{DST.stat().st_size:,} bytes")

if DST.stat().st_size != SRC.stat().st_size:
    raise RuntimeError("백업 파일 크기가 원본과 다릅니다.")

print("✓ 실제 Validation 평가 코드 Drive 영구 보존 완료")

===== evaluate.py 백업 완료 =====
원본 : /content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py
Drive: /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/reproducibility/evaluate.py
크기 : 30,451 bytes
✓ 실제 Validation 평가 코드 Drive 영구 보존 완료


In [ ]:
from google.colab import drive
from pathlib import Path
import torch
import json

drive.mount("/content/drive")

CHECKPOINT = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/"
    "checkpoints/checkpoint_best.pth"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/"
    "reproducibility"
)

OUTPUT = OUTPUT_DIR / "checkpoint_best_metadata.json"

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"checkpoint가 없습니다:\n{CHECKPOINT}"
    )

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model_state = checkpoint.get("model", {})

metadata = {
    "checkpoint_file": CHECKPOINT.name,

    "checkpoint_keys": list(checkpoint.keys()),

    "epoch_zero_based": checkpoint.get("epoch"),
    "best_epoch_one_based": (
        int(checkpoint["epoch"]) + 1
        if checkpoint.get("epoch") is not None
        else None
    ),

    "best_dice": checkpoint.get("best_dice"),
    "bad_epochs": checkpoint.get("bad_epochs"),

    "config": checkpoint.get("config"),

    # checkpoint compatibility 확인용
    # 실제 tensor 값은 저장하지 않고 이름과 shape만 저장
    "model_state": {
        key: {
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }
        for key, value in model_state.items()
    },
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with OUTPUT.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("===== Checkpoint Metadata 저장 완료 =====")
print("Checkpoint :", CHECKPOINT)
print("Metadata   :", OUTPUT)
print()
print("Checkpoint keys :", metadata["checkpoint_keys"])
print("Best Epoch      :", metadata["best_epoch_one_based"])
print("Best Dice       :", metadata["best_dice"])
print("Model tensors   :", len(metadata["model_state"]))
print()
print("✓ checkpoint 원본은 수정하지 않았습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
===== Checkpoint Metadata 저장 완료 =====
Checkpoint : /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/checkpoints/checkpoint_best.pth
Metadata   : /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S00/reproducibility/checkpoint_best_metadata.json

Checkpoint keys : ['epoch', 'model', 'optimizer', 'scheduler', 'scaler', 'best_dice', 'bad_epochs', 'config']
Best Epoch      : 104
Best Dice       : 0.7700568821869398
Model tensors   : 83

✓ checkpoint 원본은 수정하지 않았습니다.
